# BT4221 validate_cleaning_agent Notebook

## Local Resources (Windows)

### Setup spark
**Only if using local runtime!!**

In [ ]:
# # Download and install Spark
# !pip install pyspark

# # Install findspark
# !pip install findspark 

# # Initialise findspark
# import findspark
# findspark.init()

# # Install Langgraph
# !pip install -U langgraph langchain langchain-openai

# # # Set java version
# # import os
# # java_path = "C:\Program Files\Java\jdk-17.0.0.1"
# # os.environ['JAVA_HOME'] = java_path 

## Import Libraries

In [ ]:
import json
import os
from getpass import getpass
from typing import TypedDict, Optional, Literal, Any, Dict, List
from pydantic import BaseModel, Field
from openai import OpenAI
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import (
    col, year, month, dayofweek, hour, when,
    unix_timestamp, skewness, kurtosis,
    percentile_approx, count, isnan
)
from pyspark.storagelevel import StorageLevel
from langgraph.graph import StateGraph, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
import string
import time

## Create Spark Session

### Google Colab Spark Session Config

In [ ]:
# if "spark" in locals():
#     spark.stop()
#     print("Existing Spark session stopped.")

# spark = SparkSession.builder \
#     .appName("BT4221") \
#     .master("local[2]") \
#     .config("spark.driver.memory", "8g") \
#     .config("spark.driver.maxResultSize", "4g") \
#     .config("spark.sql.shuffle.partitions", "50") \
#     .config("spark.memory.fraction", "0.8") \
#     .config("spark.memory.storageFraction", "0.3") \
#     .getOrCreate()

# # Verify SparkSession created successfully
# print("App:", spark.sparkContext.appName)

### Optimised Spark Sessions Config

In [ ]:
if "spark" in locals():
    spark.stop()
    print("Existing Spark session stopped.")

# Active default Spark session (safe local baseline)
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("US_Accidents")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "48")
    .config("spark.default.parallelism", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

# Verify SparkSession created successfully
print("App:", spark.sparkContext.appName)

Existing Spark session stopped.
App: US_Accidents


### Load Dataset

In [ ]:
# # Upload CSV
# df = spark.read.csv("US_Accidents_March23.csv", header=True, inferSchema=True)

# Default parquet path from current working directory
parquet_path = "cleaned_parquet"

# Optional absolute paths
# parquet_path = "C:/Users/Kayle/OneDrive/Desktop/BT/BT4221/spark_proj/cleaned_parquet"
# parquet_path = "C:/Users/iiank/OneDrive/Documents/NUS/Y2/Y2S2/BT4221/BT4221 Project/cleaned_parquet"

df = spark.read.parquet(parquet_path)
df.show(5)

+------------+----------------+--------+-------------------+-------------------+---------+-------------------+------------+---------------+--------+--------+-----+-------+-----------+--------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+--------------+--------------+-----------------+---------------------+---------------+----------------------+---------------+---------------------+-------------+--------------+--------------------+-------------+-------------------+----------------+------------------+-------------------+
|Airport_Code|Start_Time_Month|Severity|         Start_Time|           End_Time|Start_Lat|          Start_Lng|Distance(mi)|         Street|    City|  County|State|Zipcode|   Timezone|Temperature(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|   W

## Skill.md

### Skill: validate-cleaning

In [ ]:
SKILL_PROFILE = """
---
name: validate-cleaning
description: >
  Comprehensively profile a PySpark DataFrame to surface data quality issues.
  Produces a structured JSON-like profile used by the downstream feature engineering
  agent to automate imputation, filtering, and feature selection.
mode: organisational
inputs:
  - df: PySpark DataFrame
  - target_col: Name of the target variable
  - timestamp_cols: List of columns to treat as timestamps
outputs:
  - profile: Nested dictionary of all data characteristics and recommendations
---

# Dataset Profiling Skill

## Purpose
This skill is the FIRST node in the data pipeline. It performs a thorough
examination of the input DataFrame and produces a profile dictionary.
This profile is the input to the downstream feature engineering agent.

## Key Metrics & Logic
- **Structural:** Total rows, columns, column names, and data types.
- **Missingness:** Categorizes null density from "none" to "critical."
- **Numerical:** Uses IQR for outlier detection and Skewness for imputation strategy (Mean vs Median).
- **Categorical:** Flags high cardinality and "Near-Zero Variance" (top category > 95%).
- **Temporal:** Detects "Sparse Years" (years with < 1% of total data) to identify reporting shifts.
- **Correlation:** Computes Pearson correlation to identify redundant features (Threshold > 0.9).

## Output Format
Return a single Python dictionary with this exact structure:
```python
profile = {
    "structural_overview": {
        "total_rows": int,
        "total_columns": int,
        "column_types": {
            "col_name": "numerical/categorical/timestamp/time_derived/boolean"
        }
    },
    "missing_values": {
        "col_name": {
            "null_count": int,
            "null_pct": float,
            "severity": "none/low/moderate/high/critical",
            "recommendation": str
        }
    },
    "numerical_analysis": {
        "col_name": {
            "mean": float,
            "std": float,
            "min": float,
            "max": float,
            "skewness": float,
            "kurtosis": float,
            "outlier_count": int,
            "distribution": "normal/moderate_skew/high_skew",
            "imputation_recommendation": "mean/median"
        }
    },
    "categorical_analysis": {
        "col_name": {
            "distinct_count": int,
            "top_5_values": {str: int},
            "bottom_5_values": {str: int},
            "near_zero_variance": bool,
            "encoding_recommendation": str
        }
    },
    "timestamp_analysis": {
        "col_name": {
            "min": str,
            "max": str,
            "yearly_distribution": {str: int},
            "sparse_years": [int],
            "temporal_consistency": "consistent/inconsistent"
        }
    },
    "target_analysis": {
        "column": str,
        "class_distribution": {str: int},
        "imbalance_ratio": float,
        "imbalance_classification": str,
        "smote_recommendation": str
    },
    "correlation_analysis": {
        "highly_correlated_pairs": [
            {
                "col1": str,
                "col2": str,
                "correlation": float,
                "recommendation": str
            }
        ],
        "near_duplicate_pairs": [
            {
                "col1": str,
                "col2": str,
                "correlation": float,
                "recommendation": str
            }
        ]
    },
    "quality_summary": {
        "total_issues": int,
        "critical_issues": [str],
        "warnings": [str],
        "overall_quality_score": "good/moderate/poor"
    }
}
```

## Important Notes
- This skill is PURE PYTHON AND PYSPARK — no LLM is involved
- All computations must be done using PySpark functions
- Cache the DataFrame before profiling to avoid recomputation:
  df.cache()
- The output profile is stored in AgentState["profile"]
- The downstream cleaning agent reads AgentState["profile"]
  and uses it as its SOLE INPUT for making cleaning decisions
- Do not hardcode any column names — profile ALL columns
  dynamically based on schema
- Log progress as each section completes so the user can
  track profiling progress on large datasets
"""

import os
os.makedirs("skills/profile-dataset", exist_ok=True)
with open("skills/profile-dataset/SKILL.md", "w") as f:
    f.write(SKILL_PROFILE.strip())
print("Written: skills/profile-dataset/SKILL.md")

Written: skills/profile-dataset/SKILL.md


## Previous Agent Skill (Pyspark)

### Basic Dataset Exploration

In [ ]:
# rows = df.count()
# cols = len(df.columns)

# print("Number of rows:", rows)
# print("Number of columns:", cols)

In [ ]:
# from pyspark.sql.types import IntegerType, DoubleType, FloatType, StringType, BooleanType, TimestampType

# numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, FloatType))]
# categorical_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
# boolean_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, BooleanType)]
# timestamp_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, TimestampType)]

# print(f"Numeric columns: ({len(numeric_cols)}) {numeric_cols}")
# print(f"Categorical columns: ({len(categorical_cols)}) categorical_cols")
# print(f"Boolean columns: ({len(boolean_cols)}) boolean_cols")
# print(f"Timestamp columns: ({len(timestamp_cols)}) timestamp_cols")
# print(f"Total number of columns: {len(numeric_cols) + len(categorical_cols) + len(boolean_cols) + len(timestamp_cols)}")

In [ ]:
# df.describe().show()

### Previous Pyspark Function

**This is the previous 1 function = 1 skill from work_v1.ipynb.  
The updated form of this function is below, under "Agent Skill with Helper Functions".**

In [ ]:
# def validate_cleaning(df, target_col,
#                     timestamp_cols=None):
#     """
#     Profiles a PySpark DataFrame.
#     Produces a structured dictionary consumed by the downstream feature engineering agent.
#     """

#     # Read skill file (shows marker skill is being followed)
#     with open("skills/profile-dataset/SKILL.md", "r") as f:
#         print(">> Following skill: profile-dataset/SKILL.md")

#     timestamp_cols = timestamp_cols or []
#     profile = {}

#     # Cache df to avoid recomputation
#     df.cache()
#     total_rows = df.count()
#     total_cols = len(df.columns)
#     print(f"   Profiling {total_rows:,} rows x {total_cols} columns...")

#     # --------------------------------------------------
#     # SECTION 1: Structural Overview
#     # --------------------------------------------------
#     print("\n   [Section 1] Structural overview...")

#     col_types = {}
#     numerical_cols = []
#     categorical_cols = []

#     for field in df.schema.fields:
#         dtype = str(field.dataType)
#         if any(t in dtype for t in
#                ["IntegerType", "LongType",
#                 "DoubleType", "FloatType"]):
#             col_types[field.name] = "numerical"
#             numerical_cols.append(field.name)
#         elif "TimestampType" in dtype or "DateType" in dtype:
#             col_types[field.name] = "timestamp"
#         elif "BooleanType" in dtype:
#             col_types[field.name] = "boolean"
#         else:
#             col_types[field.name] = "categorical"
#             categorical_cols.append(field.name)

#     profile["structural_overview"] = {
#         "total_rows": total_rows,
#         "total_columns": total_cols,
#         "column_names": df.columns,
#         "column_types": col_types,
#         "numerical_columns": numerical_cols,
#         "categorical_columns": categorical_cols,
#         "timestamp_columns": timestamp_cols
#     }
#     print("   ✓ [Section 1] Structural overview complete")

#     # --------------------------------------------------
#     # SECTION 2: Missing Value Analysis
#     # --------------------------------------------------
#     print("\n   [Section 2] Missing value analysis...")

#     null_counts = df.select(
#         [F.count(F.when(col(c).isNull(), c)).alias(c) for c in df.columns
#     ]).collect()[0].asDict()

#     missing_values = {}
#     for c in df.columns:
#         n = null_counts[c]
#         pct = round((n / total_rows) * 100, 4)

#         severity = "none" if pct == 0 else "low" if pct <= 5 else "moderate" if pct <= 20 else "high" if pct <= 50 else "critical"
#         recs = {
#             "none": "keep",
#             "low": "impute/drop rows",
#             "moderate": "impute",
#             "high": "check relevance",
#             "critical": "drop"}

#         missing_analysis[c] = {
#             "null_count": cnt,
#             "null_pct": pct,
#             "severity": severity,
#             "recommendation": recs[severity]
#         }

#     profile["missing_values"] = missing_values
#     print("   ✓ [Section 2] Missing value analysis complete")

#     # --------------------------------------------------
#     # SECTION 3: Numerical Column Analysis
#     # --------------------------------------------------
#     print("\n   [Section 3] Numerical column analysis...")

#     numerical_analysis = {}
#     for c in numerical_cols:
#         stats = df.select(
#             F.mean(col(c)).alias("mean"),
#             F.stddev(col(c)).alias("std"),
#             F.min(col(c)).alias("min"),
#             F.max(col(c)).alias("max"),
#             percentile_approx(col(c), 0.25).alias("q25"),
#             percentile_approx(col(c), 0.75).alias("q75"),
#             skewness(col(c)).alias("skewness"),
#             kurtosis(col(c)).alias("kurtosis")
#         ).collect()[0]

#         iqr = (stats["q75"] or 0) - (stats["q25"] or 0)
#         outliers = df.filter(
#             col(c) < (stats["q25"] or 0) - 1.5 * iqr |
#             col(c) > (stats["q75"] or 0) + 1.5 * iqr
#         ).count()

#         skew_val = abs(stats["skewness"] or 0)
#         if skew_val <= 0.5:
#             distribution = "normal"
#             impute_rec = "mean"
#         elif skew_val <= 1.0:
#             distribution = "moderate_skew"
#             impute_rec = "median"
#         else:
#             distribution = "high_skew"
#             impute_rec = "median"

#         numerical_analysis[c] = {
#             "mean": round(stats["mean"] or 0, 4),
#             "std": round(stats["std"] or 0, 4),
#             "min": round(stats["min"] or 0, 4),
#             "max": round(stats["max"] or 0, 4),
#             "q25": round(q25, 4),
#             "q75": round(q75, 4),
#             "skewness": round(stats["skewness"] or 0, 4),
#             "kurtosis": round(stats["kurtosis"] or 0, 4),
#             "outlier_count": outlier_count,
#             "distribution": distribution,
#             "imputation_recommendation": impute_rec
#         }

#     profile["numerical_analysis"] = numerical_analysis
#     print("   ✓ [Section 3] Numerical column analysis complete")

#     # --------------------------------------------------
#     # SECTION 4: Categorical Column Analysis
#     # --------------------------------------------------
#     print("\n   [Section 4] Categorical column analysis...")

#     categorical_analysis = {}
#     for c in categorical_cols:
#         distinct = df.select(c).distinct().count()

#         if distinct <= 2:
#             cardinality = "binary"
#             encoding_rec = "binary_encode"
#         elif distinct <= 10:
#             cardinality = "low"
#             encoding_rec = "one_hot_encode"
#         elif distinct <= 50:
#             cardinality = "medium"
#             encoding_rec = "label_encode or group rare categories"
#         elif distinct <= 200:
#             cardinality = "high"
#             encoding_rec = "group rare categories then label_encode"
#         else:
#             cardinality = "very_high"
#             encoding_rec = "consider dropping or heavy grouping"

#         # Top 5 values
#         top_values = (df.groupBy(c)
#                        .count()
#                        .orderBy(F.desc("count"))
#                        .limit(5)
#                        .collect())
#         top_dict = {str(r[c]): r["count"] for r in top_values}

#         # Bottom 5 values
#         bottom_values = (df.groupBy(c)
#                           .count()
#                           .orderBy("count")
#                           .limit(5)
#                           .collect())
#         bottom_dict = {str(r[c]): r["count"]
#                       for r in bottom_values}

#         # Inconsistency detection
#         values_lower = [str(r[c]).lower()
#                        for r in top_values if r[c]]
#         inconsistency = (len(values_lower) !=
#                         len(set(values_lower)))
#         inconsistent_examples = []
#         seen = {}
#         for r in top_values:
#             if r[c]:
#                 lower = str(r[c]).lower()
#                 if lower in seen:
#                     inconsistent_examples.append(
#                         f"'{seen[lower]}' and '{r[c]}'"
#                     )
#                 else:
#                     seen[lower] = r[c]

#         categorical_analysis[c] = {
#             "distinct_count": distinct,
#             "cardinality": cardinality,
#             "top_5_values": top_dict,
#             "bottom_5_values": bottom_dict,
#             "likely_identifier": distinct == total_rows,
#             "inconsistency_flag": inconsistency,
#             "inconsistent_examples": inconsistent_examples[:5],
#             "encoding_recommendation": encoding_rec
#         }

#     profile["categorical_analysis"] = categorical_analysis
#     print("   ✓ [Section 4] Categorical column analysis complete")

#     # --------------------------------------------------
#     # SECTION 5: Timestamp Analysis
#     # --------------------------------------------------
#     print("\n   [Section 5] Timestamp analysis...")

#     timestamp_analysis = {}
#     for c in timestamp_cols:
#         if c not in df.columns:
#             continue

#         ts_df = df.withColumn(c, col(c).cast("timestamp"))

#         stats = ts_df.select(
#             F.min(col(c)).alias("min_ts"),
#             F.max(col(c)).alias("max_ts")
#         ).collect()[0]

#         yearly = (ts_df.withColumn("yr", year(col(c)))
#                        .groupBy("yr").count()
#                        .orderBy("yr").collect())
#         yearly_dist = {str(r["yr"]): r["count"]
#                       for r in yearly}

#         monthly = (ts_df.withColumn("mo", month(col(c)))
#                         .groupBy("mo").count()
#                         .orderBy("mo").collect())
#         monthly_dist = {str(r["mo"]): r["count"]
#                        for r in monthly}

#         hourly = (ts_df.withColumn("hr", hour(col(c)))
#                        .groupBy("hr").count()
#                        .orderBy("hr").collect())
#         hourly_dist = {str(r["hr"]): r["count"]
#                       for r in hourly}

#         # Sparse year detection (< 1% of total rows)
#         sparse_years = [
#             int(yr) for yr, cnt in yearly_dist.items()
#             if cnt < (total_rows * 0.01)
#         ]

#         timestamp_analysis[c] = {
#             "min_timestamp": str(stats["min_ts"]),
#             "max_timestamp": str(stats["max_ts"]),
#             "yearly_distribution": yearly_dist,
#             "monthly_distribution": monthly_dist,
#             "hourly_distribution": hourly_dist,
#             "sparse_years": sparse_years,
#             "temporal_consistency": (
#                 "inconsistent" if sparse_years
#                 else "consistent"
#             )
#         }

#     profile["timestamp_analysis"] = timestamp_analysis
#     print("   ✓ Section 5 complete")

#     # --------------------------------------------------
#     # SECTION 6: Variance Analysis
#     # --------------------------------------------------
#     print("\n   [Section 6] Variance analysis...")

#     variance_analysis = {}
#     for c in df.columns:
#         distinct = (profile["categorical_analysis"]
#                    .get(c, {}).get("distinct_count") or
#                    profile["numerical_analysis"]
#                    .get(c, {}).get("distinct_count"))

#         if distinct is None:
#             distinct = df.select(c).distinct().count()

#         zero_var = distinct == 1
#         likely_id = distinct == total_rows

#         if zero_var:
#             rec = "drop — zero variance, no information"
#         elif likely_id:
#             rec = "drop — likely identifier, no predictive value"
#         else:
#             rec = "keep"

#         variance_analysis[c] = {
#             "distinct_count": distinct,
#             "zero_variance": zero_var,
#             "likely_identifier": likely_id,
#             "recommendation": rec
#         }

#     profile["variance_analysis"] = variance_analysis
#     print("   ✓ Section 6 complete")

#     # --------------------------------------------------
#     # SECTION 7: Target Column Analysis
#     # --------------------------------------------------
#     print(f"\n   [Section 7] Target column analysis ({target_col})...")

#     class_dist = (df.groupBy(target_col)
#                    .count()
#                    .orderBy(target_col)
#                    .collect())

#     class_counts = {str(r[target_col]): r["count"]
#                    for r in class_dist}
#     class_pcts = {k: round(v/total_rows*100, 4)
#                  for k, v in class_counts.items()}

#     sorted_counts = sorted(class_counts.values())
#     majority_count = sorted_counts[-1]
#     minority_count = sorted_counts[0]
#     imbalance_ratio = round(majority_count / minority_count, 2)

#     if imbalance_ratio <= 1.5:
#         imbalance_class = "balanced"
#         smote_rec = "SMOTE not needed"
#     elif imbalance_ratio <= 3.0:
#         imbalance_class = "mild_imbalance"
#         smote_rec = "SMOTE optional"
#     elif imbalance_ratio <= 10.0:
#         imbalance_class = "moderate_imbalance"
#         smote_rec = "consider SMOTE"
#     else:
#         imbalance_class = "severe_imbalance"
#         smote_rec = "strongly recommend SMOTE"

#     positive_rate = round(minority_count / total_rows, 4)
#     target_warnings = []
#     if positive_rate < 0.10:
#         target_warnings.append(
#             f"Positive class rate is {positive_rate:.2%} "
#             f"— severe imbalance detected"
#         )

#     profile["target_analysis"] = {
#         "column": target_col,
#         "class_distribution": class_counts,
#         "class_percentages": class_pcts,
#         "distinct_classes": len(class_counts),
#         "imbalance_ratio": imbalance_ratio,
#         "imbalance_classification": imbalance_class,
#         "positive_rate": positive_rate,
#         "smote_recommendation": smote_rec,
#         "warnings": target_warnings
#     }
#     print("   ✓ Section 7 complete")

#     # --------------------------------------------------
#     # SECTION 8: Correlation Analysis
#     # --------------------------------------------------
#     print("\n   [Section 8] Correlation analysis...")
#     high_corr_pairs = []

#     if len(numerical_cols) > 1:
#         for i in range(len(numerical_cols)):
#             for j in range(i + 1, len(numerical_cols)):
#                 c1, c2 = numerical_cols[i], numerical_cols[j]
#                 corr = df.stat.corr(c1, c2)
#                 if abs(corr) > 0.8:
#                     high_corr_pairs.append({
#                         "col1": c1,
#                         "col2": c2,
#                         "correlation": round(corr, 4),
#                         "recommendation": "drop one (redundant)" if abs(corr_val) > 0.95 else "consider reduction"
#                     })
#     profile["correlation_analysis"] = {"highly_correlated_pairs": high_corr_pairs}

#     print("   ✓ [Section 8] Correlation analysis complete")

#     # --------------------------------------------------
#     # SECTION 9: Quality Summary
#     # --------------------------------------------------
#     print("\n   [Section 9] Quality summary...")

#     critical_issues = []
#     warnings = []

#     # Check critical nulls
#     for c, m in profile["missing_values"].items():
#         if m["severity"] == "critical":
#             critical_issues.append(f"Critical Nulls in {c}")
#         elif m["severity"] in ["high", "moderate"]:
#             warnings.append(f"High/Moderate nulls in {c}")

#     # Check zero variance
#     for pair in high_corr_pairs:
#         if pair["correlation"] > 0.95:
#           critical.append(f"Near Duplicate: {pair['col1']} & {pair['col2']}")

#     # Check leakage
#     for c in profile["leakage_detection"]:
#         critical_issues.append(f"{c}: Data leakage risk")

#     # Check imbalance
#     if profile["target_analysis"]["imbalance_classification"] \
#        in ["severe_imbalance", "moderate_imbalance"]:
#         warnings.append(
#             f"Target imbalance: ratio = "
#             f"{profile['target_analysis']['imbalance_ratio']}"
#             f" — {profile['target_analysis']['smote_recommendation']}"
#         )

#     # Check temporal issues
#     for c, t in profile["timestamp_analysis"].items():
#         if t["temporal_consistency"] == "inconsistent":
#             warnings.append(
#                 f"{c}: sparse years detected "
#                 f"{t['sparse_years']} — consider filtering"
#             )

#     if len(critical_issues) == 0 and len(warnings) < 5:
#         quality_score = "good"
#     elif len(critical_issues) <= 3 or len(warnings) <= 10:
#         quality_score = "moderate"
#     else:
#         quality_score = "poor"

#     profile["quality_summary"] = {
#         "total_issues_found": len(critical_issues) + len(warnings),
#         "critical_issues": critical_issues,
#         "warnings": warnings,
#         "overall_quality_score": quality_score,
#         "recommended_next_steps": [
#             "1. Drop zero-variance and identifier columns",
#             "2. Drop data leakage columns",
#             "3. Filter temporally sparse rows",
#             "4. Engineer features from timestamp columns",
#             "5. Impute missing values by group",
#             "6. Standardise categorical columns",
#             "7. Apply binary encoding to Day/Night columns",
#             "8. Apply final NA drops for low-null columns",
#             "9. Apply SMOTE if target is imbalanced"
#         ]
#     }

#     print("   ✓ [Section 9] Quality summary complete")
#     print(f"\n   Overall quality: {quality_score.upper()}")
#     print(f"   Critical issues: {len(critical_issues)}")
#     print(f"   Warnings: {len(warnings)}")

#     return profile

## Agent Skill with Helper Functions

### Helper Functions

#### _is_time_derived_column

In [ ]:
def _is_time_derived_column(col_name, timestamp_cols):
    for col in timestamp_cols:
        if col_name.startswith(f"{col}_"):
            return True

    return False

#### _get_structural_overview

In [ ]:
def _get_structural_overview(df, total_rows, timestamp_cols):
    col_types = {}
    numerical_cols = []
    categorical_cols = []
    time_derived_cols = []

    for field in df.schema.fields:
        dtype = str(field.dataType)
        name = field.name

        if "TimestampType" in dtype or "DateType" in dtype or name in timestamp_cols:
            col_types[name] = "timestamp"
        elif _is_time_derived_column(name, timestamp_cols):
            col_types[name] = "time_derived"
            time_derived_cols.append(name)
        elif "BooleanType" in dtype:
            col_types[name] = "boolean"
        elif any(t in dtype for t in ["IntegerType", "LongType", "DoubleType", "FloatType"]):
            col_types[name] = "numerical"
            numerical_cols.append(name)
        else:
            col_types[name] = "categorical"
            categorical_cols.append(name)

    overview = {
        "total_rows": total_rows,
        "total_columns": len(df.columns),
        "column_types": col_types
    }
    return overview, numerical_cols, categorical_cols, time_derived_cols

#### _profile_missing_values

In [ ]:
def _profile_missing_values(df, total_rows):
    null_exprs = []

    for field in df.schema:
        name = field.name
        dtype = field.dataType

        condition = F.col(name).isNull()
        if isinstance(dtype, (T.DoubleType, T.FloatType)):
            condition = condition | F.isnan(F.col(name))

        null_exprs.append(F.count(F.when(condition, name)).alias(name))

    null_counts = df.select(null_exprs).collect()[0].asDict()

    missing_analysis = {}
    recs = {
        "none": "keep",
        "low": "impute/drop rows",
        "moderate": "impute",
        "high": "check relevance",
        "critical": "drop"
    }

    for c in df.columns:
        n = null_counts[c]
        pct = round((n / total_rows) * 100, 4)

        if pct == 0: severity = "none"
        elif pct <= 5: severity = "low"
        elif pct <= 20: severity = "moderate"
        elif pct <= 50: severity = "high"
        else: severity = "critical"

        missing_analysis[c] = {
            "null_count": n,
            "null_pct": pct,
            "severity": severity,
            "recommendation": recs[severity]
        }

    return missing_analysis

#### _profile_numerical_columns

In [ ]:
from pyspark.sql import functions as F

def _profile_numerical_columns(df, columns, missing_values):
    results = {}

    stats_exprs = []
    for c in columns:
        stats_exprs.extend([
            F.mean(c).alias(f"{c}_mean"),
            F.stddev(c).alias(f"{c}_std"),
            F.min(c).alias(f"{c}_min"),
            F.max(c).alias(f"{c}_max"),
            F.percentile_approx(c, [0.25, 0.75]).alias(f"{c}_q"),
            F.skewness(c).alias(f"{c}_skew"),
            F.kurtosis(c).alias(f"{c}_kurt")
        ])

    # single pass agg
    all_stats = df.select(stats_exprs).collect()[0].asDict()

    outlier_exprs = []
    processed_stats = {}

    for c in columns:
        m = all_stats[f"{c}_mean"]
        s = all_stats[f"{c}_std"]
        mn = all_stats[f"{c}_min"]
        mx = all_stats[f"{c}_max"]
        q_list = all_stats[f"{c}_q"]
        sk = all_stats[f"{c}_skew"]
        kt = all_stats[f"{c}_kurt"]

        q25, q75 = q_list[0], q_list[1]
        iqr = q75 - q25

        lower_bound = q25 - 1.5 * iqr
        upper_bound = q75 + 1.5 * iqr

        # count outliers without a filter
        if iqr == 0:
            outlier_exprs.append(F.lit(0).alias(f"{c}_outliers"))
        else:
            outlier_exprs.append(
                F.count(F.when((F.col(c) < lower_bound) | (F.col(c) > upper_bound), c)).alias(f"{c}_outliers")
            )

        processed_stats[c] = {
            "m": m, "s": s, "min": mn, "max": mx, "sk": sk, "kt": kt, "iqr": iqr
        }

    # single pass for outliers
    all_outliers = df.select(outlier_exprs).collect()[0].asDict()

    for c in columns:
        stats = processed_stats[c]
        skew_val = abs(stats["sk"] or 0)

        if skew_val <= 0.5:
            distribution = "normal"
            impute_rec = "mean"
        elif skew_val <= 1.0:
            distribution = "moderate_skew"
            impute_rec = "median"
        else:
            distribution = "high_skew"
            impute_rec = "median"

        # if no missing values,rec = not needed
        if missing_values[c]["null_count"] == 0:
            impute_rec = "not needed"

        results[c] = {
            "mean": round(stats["m"] or 0, 4),
            "std": round(stats["s"] or 0, 4),
            "min": round(stats["min"] or 0, 4),
            "max": round(stats["max"] or 0, 4),
            "skewness": round(stats["sk"] or 0, 4),
            "kurtosis": round(stats["kt"] or 0, 4),
            "outlier_count": all_outliers[f"{c}_outliers"],
            "distribution": distribution,
            "imputation_recommendation": impute_rec
        }

    return results

#### _profile_categorical_columns

In [ ]:
def _profile_categorical_columns(df, columns, total_rows):
    results = {}

    # Columns known to contain repeating semantic sub-tokens
    discovery_targets = ["Street", "Weather_Condition"]
    stop_words = {"", "n", "s", "e", "w", "and", "the", "with", "near", "at", "by", "of", "in", "to", "for", "from", "mostly", "partly", "with"}

    for c in columns:
        # Standard profiling
        distinct_cnt = df.select(c).distinct().count()
        top_values = df.groupBy(c).count().orderBy(F.desc("count")).limit(5).collect()
        top_dict = {str(r[c]): r["count"] for r in top_values}
        top_count = list(top_dict.values())[0] if top_dict else 0

        # Automated token discovery
        extraction_hint = None

        if c in discovery_targets:
            tokens_df = df.select(F.explode(F.split(F.lower(F.col(c)), r"[\s,/-]+")).alias("token")) \
                          .withColumn("token", F.regexp_replace(F.col("token"), r"[^a-z0-9]", ""))

            discovered_tokens = tokens_df.groupBy("token").count() \
                                         .orderBy(F.desc("count")) \
                                         .limit(20) \
                                         .collect()
            keywords = [
                r["token"] for r in discovered_tokens
                if r["token"] not in stop_words and not r["token"].isdigit()
            ][:10]

            extraction_hint = {
                "type": "string_keyword_extraction",
                "discovered_patterns": keywords,
                "coverage_pct": round((top_count / total_rows) * 100, 2)
            }

        if distinct_cnt <= 10:
            rec = "one_hot_encode"
        elif extraction_hint:
            rec = "extract_binary_features"
        else:
            rec = "group_rare_categories_and_label_encode"

        results[c] = {
            "distinct_count": distinct_cnt,
            "top_5_values": top_dict,
            "near_zero_variance": (top_count / total_rows) > 0.95,
            "extraction_hint": extraction_hint,
            "encoding_recommendation": rec
        }

    return results

#### _profile_timestamps

In [ ]:
def _profile_timestamps(df, columns, total_rows):
    results = {}
    for c in columns:
        temp_df = df.withColumn("_ts", F.to_timestamp(col(c)))

        stats = temp_df.select(
            F.min("_ts").alias("min_ts"),
            F.max("_ts").alias("max_ts")
        ).collect()[0]

        yearly = temp_df.withColumn("year", F.year("_ts")).groupBy("year").count().collect()
        yearly_dist = {str(r["year"]): r["count"] for r in yearly}

        # sparse = < 1% total rows
        sparse = [yr for yr, count in yearly_dist.items() if count < (total_rows * 0.01)]

        results[c] = {
            "min": str(stats["min_ts"]),
            "max": str(stats["max_ts"]),
            "yearly_distribution": yearly_dist,
            "sparse_years": [int(yr) for yr in sparse],
            "temporal_consistency": "inconsistent" if sparse else "consistent"
        }
    return results

#### _profile_target

In [ ]:
def _profile_target(df, target_col, total_rows):
    dist = df.groupBy(target_col).count().collect()
    counts = {str(r[target_col]): r["count"] for r in dist}

    class_pcts = {k: round(v/total_rows*100, 4) for k, v in counts.items()}

    vals = list(counts.values())
    imbalance_ratio = max(vals) / min(vals) if min(vals) > 0 else 0

    if imbalance_ratio > 10:
        classification = "severe_imbalance"
        smote_rec = "strongly recommend SMOTE"
    elif imbalance_ratio > 3:
        classification = "moderate_imbalance"
        smote_rec = "consider SMOTE"
    else:
        classification = "balanced"
        smote_rec = "not needed"

    return {
        "column": target_col,
        "class_distribution": counts,
        "imbalance_ratio": round(imbalance_ratio, 2),
        "imbalance_classification": classification,
        "smote_recommendation": smote_rec
    }

#### _profile_correlations

In [ ]:
def _profile_correlations(df, numerical_cols):
    high_corr_pairs = []
    near_duplicate_pairs = []
    # limit to 20 if many numerical cols
    cols_to_check = numerical_cols[:20]

    for i in range(len(cols_to_check)):
        for j in range(i + 1, len(cols_to_check)):
            c1, c2 = cols_to_check[i], cols_to_check[j]
            correlation = df.stat.corr(c1, c2)

            if abs(correlation) > 0.95:
                near_duplicate_pairs.append({
                    "col1": c1,
                    "col2": c2,
                    "correlation": round(correlation, 4),
                    "recommendation": "drop one"
                })
            elif abs(correlation) > 0.85:
                high_corr_pairs.append({
                    "col1": c1,
                    "col2": c2,
                    "correlation": round(correlation, 4),
                    "recommendation": "drop one" if abs(correlation) > 0.95 else "feature reduction"
                })
    return {"highly_correlated_pairs": high_corr_pairs, "near_duplicate_pairs": near_duplicate_pairs}

#### _generate_quality_summary

In [ ]:
def _generate_quality_summary(profile):
    critical_issues = []
    warnings = []

    # Check for critical nulls
    for c, data in profile["missing_values"].items():
        if data["severity"] == "critical":
            critical_issues.append(f"Critical Nulls: {c}")

    # Determine score
    if not critical_issues and len(warnings) < 5:
      score = "good"
    elif len(critical_issues) <= 3 or len(warnings) <= 10:
      score = "moderate"
    else:
      score = "poor"

    return {
        "total_issues": len(critical_issues) + len(warnings),
        "critical_issues": critical_issues,
        "warnings": warnings,
        "overall_quality_score": score
    }

#### _generate_agent_recommendations

In [ ]:
def _generate_agent_recommendations(profile):
    # flatten nested dicts
    cleaning_rec = []
    feature_eng_rec = []

    for col, data in profile["missing_values"].items():
        if data["severity"] == "critical":
            cleaning_rec.append({"action": "drop_column", "column": col, "reason": "critical_null_density"})
        elif data["severity"] in ["low", "moderate"]:
            cleaning_rec.append({
                "action": "impute",
                "column": col,
                "strategy": profile["numerical_analysis"].get(col, {}).get("imputation_recommendation", "mode")
            })

    for col, data in profile["categorical_analysis"].items():
        if data.get("extraction_hint"):
            feature_eng_rec.append({
                "action": "extract_binary_features",
                "column": col,
                "logic": data["extraction_hint"]["type"],
                "keywords": data["extraction_hint"]["discovered_patterns"],
                "reason": "High cardinality with repeating semantic tokens"
            })
        else:
            feature_eng_rec.append({
                "action": "encode",
                "column": col,
                "method": data["encoding_recommendation"]
            })

    for pair in profile["correlation_analysis"].get("near_duplicate_pairs", []):
        cleaning_rec.append({"action": "drop_column", "column": pair["col2"], "reason": f"redundant_with_{pair['col1']}"})

    for col, data in profile["categorical_analysis"].items():
        if not data.get("zero_variance") and not data.get("likely_identifier"):
            feature_eng_rec.append({"action": "encode", "column": col, "method": data["encoding_recommendation"]})

    target = profile["target_analysis"]
    if target["imbalance_classification"] in ["severe_imbalance", "moderate_imbalance"]:
        feature_eng_rec.append({"action": "balance_target", "method": "SMOTE", "ratio": target["imbalance_ratio"]})

    return cleaning_rec, feature_eng_rec

### Coordinator Function

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

def validate_cleaning(df, target_col, timestamp_cols=None):
    """
    Coordinator function that profiles a PySpark DataFrame.
    Produces a structured dictionary consumed by the downstream feature
    engineering agent.
    """
    timestamp_cols = timestamp_cols or []
    profile = {}

    # Initial setup
    df.cache()
    total_rows = df.count()
    total_cols = len(df.columns)
    print(f"Profiling {total_rows:,} rows x {total_cols} columns...")

    # Sections as described in skill.md
    profile["structural_overview"], numerical_cols, categorical_cols, time_derived_cols = _get_structural_overview(df, total_rows, timestamp_cols)
    print("✓ Section 1: Structural overview complete")

    profile["missing_values"] = _profile_missing_values(df, total_rows)
    print("✓ Section 2: Missing value analysis complete")

    profile["numerical_analysis"] = _profile_numerical_columns(df, numerical_cols, profile["missing_values"])
    print("✓ Section 3: Numerical column analysis complete")

    profile["categorical_analysis"] = _profile_categorical_columns(df, categorical_cols, total_rows)
    print("✓ Section 4: Categorical column analysis complete")

    profile["timestamp_analysis"] = _profile_timestamps(df, timestamp_cols, total_rows)
    print("✓ Section 5: Timestamp analysis complete")

    profile["target_analysis"] = _profile_target(df, target_col, total_rows)
    print("✓ Section 6: Target column analysis complete")

    profile["correlation_analysis"] = _profile_correlations(df, numerical_cols)
    print("✓ Section 7: Correlation analysis complete")

    cleaning_rec, feature_eng_rec = _generate_agent_recommendations(profile)
    print("✓ Section 8: Generate agent recommendations complete")

    # Quality Summary
    profile["quality_summary"] = _generate_quality_summary(profile)
    print(f"✓ Profile complete. Quality Score: {profile['quality_summary']['overall_quality_score'].upper()}")

    df.unpersist()
    return {
        "profile": profile,
        "cleaning_rec": cleaning_rec,
        "feature_eng_rec": feature_eng_rec,
        "quality_score": profile["quality_summary"]["overall_quality_score"]
    }

## Create Agent State

In [ ]:
# Keep Spark DataFrame outside LangGraph state because checkpointer requires serializable state.
RUNTIME_DF = df

class AgentState(TypedDict):
    profile: Optional[dict]                 # output of profile skill
    cleaning_plan: Optional[dict]           # optional future cleaning plan
    validation_status: Optional[str]        # approved | reclean_requested | stopped
    validation_message: Optional[str]       # details for final decision
    evaluator_verdict: Optional[dict]       # evaluator decision payload
    human_decision: Optional[str]           # approve / reclean / stop
    execution_log: list                     # step-by-step trace

## Create validate_cleaning_node

In [ ]:
def validate_cleaning_node(state: AgentState) -> AgentState:
    print("=" * 55)
    print(">> NODE 1: Running dataset profiling skill...")
    print("=" * 55)

    # Use runtime DataFrame reference (not stored in graph state).
    profile = validate_cleaning(
        df=RUNTIME_DF,
        target_col="Severity",
        timestamp_cols=["Start_Time", "End_Time"]
    )

    state["profile"] = profile
    state.setdefault("execution_log", []).append(
        {
            "node": "profile",
            "quality_score": profile.get("quality_score"),
            "critical_issues": len(profile.get("profile", {}).get("quality_summary", {}).get("critical_issues", [])),
            "total_issues": profile.get("profile", {}).get("quality_summary", {}).get("total_issues", 0)
        }
    )

    print("\n   Profile stored in AgentState['profile']")
    print("   Ready for evaluator + human review.")
    return state

## Build Agentic Data Pipeline

In [ ]:
Decision = Literal["ready", "needs_more_cleaning"]
Priority = Literal["low", "medium", "high", "critical"]

class Concern(BaseModel):
    area: str = Field(..., description="Area of concern")
    issue: str
    why_it_matters: str
    suggested_action: str
    priority: Priority


class CleaningAgentVerdict(BaseModel):
    decision: Decision
    confidence: float = Field(..., ge=0.0, le=1.0)
    strengths: List[str] = Field(default_factory=list)
    concerns: List[Concern] = Field(default_factory=list)
    next_steps_prioritized: List[str] = Field(default_factory=list)
    summary_for_team: str


class LLMCleaningEvaluatorAgent:
    def __init__(self, api_key: str, model_name: str = "gpt-4o-mini"):
        self.client = OpenAI(api_key=api_key)
        self.model_name = model_name

    def _extract_json_payload(self, raw_text: str) -> Dict[str, Any]:
        text = (raw_text or "").strip()
        if text.startswith("```"):
            lines = text.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].startswith("```"):
                lines = lines[:-1]
            text = "\n".join(lines).strip()

        try:
            return json.loads(text)
        except Exception:
            start = text.find("{")
            end = text.rfind("}")
            if start != -1 and end != -1 and end > start:
                return json.loads(text[start : end + 1])
            raise

    def _normalize_payload(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        payload = dict(payload or {})
        if payload.get("decision") not in {"ready", "needs_more_cleaning"}:
            payload["decision"] = "needs_more_cleaning"

        try:
            confidence = float(payload.get("confidence", 0.5))
        except Exception:
            confidence = 0.5
        payload["confidence"] = max(0.0, min(1.0, confidence))

        if not isinstance(payload.get("strengths"), list):
            payload["strengths"] = []

        concerns = payload.get("concerns", [])
        if not isinstance(concerns, list):
            concerns = []
        cleaned_concerns = []
        valid_priorities = {"low", "medium", "high", "critical"}
        for item in concerns:
            if not isinstance(item, dict):
                continue
            priority = str(item.get("priority", "medium")).lower().strip()
            if priority not in valid_priorities:
                priority = "medium"
            cleaned_concerns.append(
                {
                    "area": str(item.get("area", "general_data_quality")).strip() or "general_data_quality",
                    "issue": str(item.get("issue", "Potential data quality concern identified")).strip(),
                    "why_it_matters": str(
                        item.get(
                            "why_it_matters",
                            "This may reduce model reliability or generalization.",
                        )
                    ).strip(),
                    "suggested_action": str(
                        item.get("suggested_action", "Review and apply targeted cleaning.")
                    ).strip(),
                    "priority": priority,
                }
            )
        payload["concerns"] = cleaned_concerns

        steps = payload.get("next_steps_prioritized", [])
        if not isinstance(steps, list):
            steps = []
        payload["next_steps_prioritized"] = [str(s) for s in steps][:10]

        summary = payload.get("summary_for_team")
        if not isinstance(summary, str) or not summary.strip():
            summary = "Evaluator produced a provisional verdict. Review concerns and next steps."
        payload["summary_for_team"] = summary.strip()
        return payload

    def evaluate(self, evaluation_context: Dict[str, Any]) -> CleaningAgentVerdict:
        system_prompt = (
            "You are a senior ML data quality reviewer. "
            "Evaluate whether current dataset quality is good enough for modeling. "
            "Return ONLY valid JSON matching the requested schema."
        )

        user_prompt = {
            "task": "Decide if cleaning is sufficient and what should happen next.",
            "required_output_schema": {
                "decision": "ready | needs_more_cleaning",
                "confidence": "float between 0 and 1",
                "strengths": ["string"],
                "concerns": [
                    {
                        "area": "string",
                        "issue": "string",
                        "why_it_matters": "string",
                        "suggested_action": "string",
                        "priority": "low|medium|high|critical"
                    }
                ],
                "next_steps_prioritized": ["string"],
                "summary_for_team": "string"
            },
            "evaluation_context": evaluation_context,
        }

        response = self.client.chat.completions.create(
            model=self.model_name,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_prompt)},
            ],
        )

        content = response.choices[0].message.content
        payload = self._extract_json_payload(content)
        payload = self._normalize_payload(payload)
        return CleaningAgentVerdict.model_validate(payload)


def build_evaluation_context(state: AgentState) -> Dict[str, Any]:
    prof = state["profile"]["profile"]
    quality = prof.get("quality_summary", {})
    return {
        "quality_score": state["profile"].get("quality_score"),
        "missing_values": prof.get("missing_values", {}),
        "target_analysis": prof.get("target_analysis", {}),
        "correlation_analysis": prof.get("correlation_analysis", {}),
        "categorical_analysis": prof.get("categorical_analysis", {}),
        "quality_summary": quality,
        "proposed_cleaning_actions": state["profile"].get("cleaning_rec", [])[:20],
        "proposed_feature_actions": state["profile"].get("feature_eng_rec", [])[:20],
    }


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY: ").strip()
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required for evaluator node.")

EVALUATOR_MODEL_NAME = "gpt-4o-mini"
llm_evaluator = LLMCleaningEvaluatorAgent(
    api_key=OPENAI_API_KEY,
    model_name=EVALUATOR_MODEL_NAME,
)


def evaluator_node(state: AgentState) -> AgentState:
    context = build_evaluation_context(state)
    verdict = llm_evaluator.evaluate(context).model_dump()

    state["evaluator_verdict"] = verdict
    state.setdefault("execution_log", []).append({"node": "evaluator", **verdict})

    print("=" * 55)
    print(">> NODE 2: LLM evaluator verdict")
    print("=" * 55)
    print(json.dumps(verdict, indent=2))
    return state


def human_review_node(state: AgentState):
    """Human-in-the-loop gate using LangGraph interrupt."""
    verdict = state.get("evaluator_verdict", {})
    payload = {
        "message": "Approve current dataset for modeling?",
        "allowed_decisions": ["approve", "reclean", "stop"],
        "default": "reclean",
        "verdict": verdict,
        "instruction": "Return one of: approve | reclean | stop"
    }

    human_decision = str(interrupt(payload)).strip().lower()
    if human_decision not in {"approve", "reclean", "stop"}:
        human_decision = "reclean"

    state["human_decision"] = human_decision
    state.setdefault("execution_log", []).append({"node": "human_review", "decision": human_decision})
    return state


def route_after_human(state: AgentState) -> str:
    decision = state.get("human_decision", "reclean")

    if decision == "approve":
        state["validation_status"] = "approved"
        state["validation_message"] = "Human approved dataset for modeling."
    elif decision == "stop":
        state["validation_status"] = "stopped"
        state["validation_message"] = "Human stopped pipeline."
    else:
        # Manual handoff mode: human cleans outside this graph and reruns later.
        state["validation_status"] = "reclean_requested"
        state["validation_message"] = "Human requested manual reclean. End run and rerun on updated dataset."

    return "end"


def build_pipeline():
    graph = StateGraph(AgentState)

    graph.add_node("profile", validate_cleaning_node)
    graph.add_node("evaluator", evaluator_node)
    graph.add_node("human_review", human_review_node)

    graph.set_entry_point("profile")
    graph.add_edge("profile", "evaluator")
    graph.add_edge("evaluator", "human_review")
    graph.add_conditional_edges(
        "human_review",
        route_after_human,
        {
            "end": END
        },
    )

    return graph.compile(checkpointer=MemorySaver())


initial_state = AgentState(
    profile=None,
    cleaning_plan=None,
    validation_status=None,
    validation_message=None,
    evaluator_verdict=None,
    human_decision=None,
    execution_log=[],
)

pipeline = build_pipeline()
config = {"configurable": {"thread_id": "hitl-cleaning-review-llm-1"}}

# First invoke pauses at human_review_node and returns an interrupt payload.
first_result = pipeline.invoke(initial_state, config=config)
print("Paused for human review. Interrupt payload:")
print(first_result.get("__interrupt__", first_result))

>> NODE 1: Running dataset profiling skill...


Profiling 5,270,673 rows x 50 columns...
✓ Section 1: Structural overview complete


✓ Section 2: Missing value analysis complete


✓ Section 3: Numerical column analysis complete


✓ Section 4: Categorical column analysis complete
✓ Section 5: Timestamp analysis complete
✓ Section 6: Target column analysis complete
✓ Section 7: Correlation analysis complete
✓ Section 8: Generate agent recommendations complete
✓ Profile complete. Quality Score: GOOD

   Profile stored in AgentState['profile']
   Ready for evaluator + human review.
>> NODE 2: LLM evaluator verdict
{
  "decision": "needs_more_cleaning",
  "confidence": 0.85,
  "strengths": [
    "No missing values across all columns.",
    "Overall quality score is good."
  ],
  "concerns": [
    {
      "area": "Imbalance in target variable",
      "issue": "Severe class imbalance in the 'Severity' column.",
      "why_it_matters": "Imbalance can lead to biased models that perform poorly on minority classes.",
      "suggested_action": "Apply SMOTE to balance the classes.",
      "priority": "high"
    },
    {
      "area": "Redundant features",
      "issue": "Presence of redundant duration columns.",
      "why_

## Human Decision Step (Manual Resume Required)
After running the previous cell, the graph pauses at `human_review_node` and waits.
Set one decision in the next cell: `approve`, `reclean`, or `stop`.

Status outcomes:
- `approve` -> `validation_status = approved`
- `reclean` -> `validation_status = reclean_requested` (pipeline ends, clean manually, then rerun)
- `stop` -> `validation_status = stopped`

In [ ]:
# Choose exactly one: "approve", "reclean", "stop"
HUMAN_DECISION = "approve"  # set this manually before running

if HUMAN_DECISION not in {"approve", "reclean", "stop"}:
    raise ValueError("Set HUMAN_DECISION to one of: approve, reclean, stop")

resume_result = pipeline.invoke(Command(resume=HUMAN_DECISION), config=config)

print("validation_status:", resume_result.get("validation_status"))
print("validation_message:", resume_result.get("validation_message"))
print("human_decision:", resume_result.get("human_decision"))
print("\nExecution log:")
for row in resume_result.get("execution_log", []):
    print(row)

validation_status: None
validation_message: None
human_decision: approve

Execution log:
{'node': 'profile', 'quality_score': 'good', 'critical_issues': 0, 'total_issues': 0}
{'node': 'evaluator', 'decision': 'needs_more_cleaning', 'confidence': 0.85, 'strengths': ['No missing values across all columns.', 'Overall quality score is good.'], 'concerns': [{'area': 'Imbalance in target variable', 'issue': "Severe class imbalance in the 'Severity' column.", 'why_it_matters': 'Imbalance can lead to biased models that perform poorly on minority classes.', 'suggested_action': 'Apply SMOTE to balance the classes.', 'priority': 'high'}, {'area': 'Redundant features', 'issue': 'Presence of redundant duration columns.', 'why_it_matters': 'Redundant features can lead to increased model complexity and overfitting.', 'suggested_action': "Drop 'Duration_Minutes' and 'Duration_Hours' columns.", 'priority': 'medium'}], 'next_steps_prioritized': ['Apply SMOTE to address class imbalance.', 'Drop redundant

# Stop Spark

In [ ]:
spark.stop()